# Fire U-Net training setup

This notebook prepares the Landsat-8 Active Fire data for U-Net training: the training index, the train/val/test split, and the PyTorch data loader. Training is not started here yet.

It runs in two places:

- **Kaggle.** The full North America datasets are mounted under `/kaggle/input/`, so nothing is downloaded. Use this for the real index and for training.
- **Locally.** It downloads a few path/row folders through the Kaggle API into `data/kaggle/`, so the code can be checked on a laptop.

Data: [North America 1](https://www.kaggle.com/datasets/pe4eniks/fire-segmentation-north-america-1) (19 GB) and [North America 2](https://www.kaggle.com/datasets/pe4eniks/fire-segmentation-north-america-2) (4.4 GB), a CC0 re-upload of [Pereira et al. 2021](https://github.com/pereira-gha/activefire).

## Settings

`TARGET` is the mask the U-Net learns. The voting mask marks a pixel as fire when at least two of the three algorithms agree. The authors' best U-Net was trained on it.

In [ ]:
import contextlib
import io
import random
import re
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import tifffile
except ImportError:
    %pip install -q tifffile
    import tifffile

DATASETS = (
    "pe4eniks/fire-segmentation-north-america-1",
    "pe4eniks/fire-segmentation-north-america-2",
)

# Landsat band in each file channel. Band 8, the panchromatic band, is not in the files.
FILE_BANDS = (1, 2, 3, 4, 5, 6, 7, 9, 10, 11)
# Model input, the same three bands as the paper's 3-channel U-Nets.
INPUT_BANDS = (7, 6, 2)
MAX_PIXEL_VALUE = 65535

# Algorithm masks are in {path_row}_masks/, combined masks in {path_row}_masks_derivates/.
# A missing mask file means that source found no fire in the patch.
MASK_SOURCES = ("Schroeder", "Murphy", "Kumar-Roy", "voting", "intersection")
DERIVED_SOURCES = ("voting", "intersection")
TARGET = "voting"

SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42

# Local runs download this many path/row folders from North America 2.
LOCAL_PATH_ROWS = 7

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working")
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    LOCAL_DATA = REPO / "data" / "kaggle"
    # Kept apart from data/training, where real Kaggle outputs are downloaded.
    OUTPUT_DIR = REPO / "data" / "training" / "local"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Running on", "Kaggle" if ON_KAGGLE else "this machine")

## Kaggle access

On Kaggle, add both datasets to the notebook with **Add Input**. Locally, the Kaggle API reads `~/.kaggle/kaggle.json`.

In [ ]:
def find_kaggle_root(slug):
    """Find a mounted dataset. Kaggle has used more than one mount layout."""
    for pattern in (slug, f"*/{slug}", f"*/*/{slug}"):
        matches = sorted(Path("/kaggle/input").glob(pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Add {slug} to this notebook with Add Input")


if ON_KAGGLE:
    DATA_ROOTS = {ref: find_kaggle_root(ref.split("/")[1]) for ref in DATASETS}
else:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    print("Kaggle user:", api.get_config_value("username"))
    DATA_ROOTS = {ref: LOCAL_DATA / ref.split("/")[1] for ref in DATASETS}

for ref, root in DATA_ROOTS.items():
    print(ref, "->", root)

## Local sample

Skipped on Kaggle. Downloads every file in the first few North America 2 path/row folders that have voting masks, keeping the Kaggle folder layout so the rest of the notebook reads them the same way it reads the Kaggle mount.

In [ ]:
def path_row_folder(name):
    return re.sub(r"_masks(_derivates)?$", "", name.split("/")[0])


def with_retry(call, attempts=6):
    """Kaggle answers 429 when it gets too many requests, so wait and retry."""
    for attempt in range(attempts):
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                return call()
        except Exception as error:
            if "429" not in str(error) or attempt == attempts - 1:
                raise
            time.sleep(10 * 2**attempt)


def list_dataset_files(ref):
    names, token = [], None
    while True:
        page = with_retry(lambda: api.dataset_list_files(ref, page_token=token, page_size=1000))
        names += [file.name for file in page.files]
        token = page.next_page_token
        if not token:
            return names


def download_local_sample(ref, path_row_count):
    root = DATA_ROOTS[ref]
    existing = sorted(path.name for path in root.glob("z*") if re.fullmatch(r"z\d{6}", path.name))
    if len(existing) >= path_row_count:
        print(f"Using {len(existing)} downloaded path/row folders in {root}")
        return
    names = list_dataset_files(ref)
    with_voting = sorted({path_row_folder(name) for name in names if "_voting_" in name})
    chosen = set(with_voting[:path_row_count])
    wanted = [name for name in names if path_row_folder(name) in chosen]
    for name in wanted:
        target = root / name
        if target.exists():
            continue
        # Kaggle sends some files, the images here, as {name}.zip.
        zipped = target.with_name(target.name + ".zip")
        if not zipped.exists():
            with_retry(lambda: api.dataset_download_file(ref, name, path=root, quiet=True))
        if zipped.exists():
            with zipfile.ZipFile(zipped) as archive:
                target.write_bytes(archive.read(archive.namelist()[0]))
            zipped.unlink()
    print(f"{len(wanted)} files from {sorted(chosen)} in {root}")


if not ON_KAGGLE:
    download_local_sample(DATASETS[1], LOCAL_PATH_ROWS)

## Training index

One row per image patch, saved as `index.csv`. File paths are relative to the dataset root, so the same index works on any Kaggle notebook that adds the same datasets.

Every image patch here was flagged by at least one algorithm. A patch is **no-fire** for the target when the target mask file is missing, which happens when the algorithms disagree.

Fire pixel counts are stored for every mask source, so `TARGET` can change without rebuilding the index.

The split is by Landsat path/row, not by patch. The same path/row appears on several dates with the same patch numbers, so a patch split would put the same ground in train and val.

In [ ]:
PATCH_NAME = re.compile(
    r"^(?P<scene>LC08_\w{4}_(?P<path_row>\d{6})_(?P<acquired>\d{8})_\d{8}_\d{2}_\w{2})_(?P<patch>p\d+)$"
)
IMAGE_FOLDER = re.compile(r"^z\d{6}$")


def tif_names(folder):
    return {path.name for path in folder.glob("*.tif")} if folder.is_dir() else set()


def count_fire(path):
    return int((tifffile.imread(path) == 1).sum())


def build_index():
    """Every file access on the Kaggle input mount is slow, so each mask folder is listed
    once instead of checking files one by one, and masks are read in parallel."""
    rows, masks_to_read = [], []
    for ref, root in DATA_ROOTS.items():
        if not root.exists():
            continue
        for folder in sorted(path for path in root.iterdir() if IMAGE_FOLDER.match(path.name)):
            present = {suffix: tif_names(root / f"{folder.name}{suffix}") for suffix in ("_masks", "_masks_derivates")}
            for image_path in sorted(folder.glob("*.tif")):
                match = PATCH_NAME.match(image_path.stem)
                row = {
                    "dataset": ref,
                    "image_file": str(image_path.relative_to(root)),
                    "scene_id": match["scene"],
                    "path_row": match["path_row"],
                    "acquired": match["acquired"],
                    "patch_id": match["patch"],
                }
                for source in MASK_SOURCES:
                    suffix = "_masks_derivates" if source in DERIVED_SOURCES else "_masks"
                    name = f"{match['scene']}_{source}_{match['patch']}.tif"
                    found = name in present[suffix]
                    row[f"{source}_file"] = f"{folder.name}{suffix}/{name}" if found else ""
                    row[f"{source}_fire_pixels"] = 0
                    if found:
                        masks_to_read.append((len(rows), source, root / row[f"{source}_file"]))
                rows.append(row)
    if not rows:
        raise FileNotFoundError(f"No image patches under {list(DATA_ROOTS.values())}")
    print(f"{len(rows)} patches, reading {len(masks_to_read)} masks")
    with ThreadPoolExecutor(max_workers=16) as pool:
        counts = pool.map(count_fire, [path for _, _, path in masks_to_read])
        for (row_number, source, _), count in zip(masks_to_read, counts):
            rows[row_number][f"{source}_fire_pixels"] = count
    return pd.DataFrame(rows)


def assign_splits(index):
    path_rows = sorted(index.path_row.unique())
    random.Random(SEED).shuffle(path_rows)
    train_end = round(len(path_rows) * SPLIT_FRACTIONS["train"])
    val_end = train_end + round(len(path_rows) * SPLIT_FRACTIONS["val"])
    split_of = {}
    for position, path_row in enumerate(path_rows):
        split_of[path_row] = "train" if position < train_end else "val" if position < val_end else "test"
    return index.assign(split=index.path_row.map(split_of))


index = assign_splits(build_index())
index["classifier_label"] = np.where(index[f"{TARGET}_fire_pixels"] > 0, "fire", "no-fire")
index_path = OUTPUT_DIR / "index.csv"
index.to_csv(index_path, index=False)
print(f"{len(index)} patches written to {index_path}")
index.head()

In [ ]:
summary = index.groupby("split").agg(
    path_rows=("path_row", "nunique"),
    patches=("image_file", "size"),
    fire_patches=("classifier_label", lambda labels: int((labels == "fire").sum())),
)
summary["fire_share"] = (summary.fire_patches / summary.patches).round(3)
print(f"Target mask: {TARGET}")
display(summary.reindex([split for split in SPLIT_FRACTIONS if split in summary.index]))
pd.Series(
    {source: int((index[f"{source}_fire_pixels"] > 0).sum()) for source in MASK_SOURCES},
    name="patches_with_fire",
)

## PyTorch data loader

Each item is the image as `(3, 256, 256)` float32 in 0 to 1, and the target mask as `(1, 256, 256)` float32 of 0 and 1. A patch with no target mask file gets an all-zero mask.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset

CHANNELS = [FILE_BANDS.index(band) for band in INPUT_BANDS]


def read_patch(row, target=TARGET):
    """The input bands as uint16 (3, 256, 256) and the target mask as bool (256, 256)."""
    root = DATA_ROOTS[row.dataset]
    image = np.ascontiguousarray(tifffile.imread(root / row.image_file)[:, :, CHANNELS].transpose(2, 0, 1))
    # Empty paths come back as NaN when index.csv is reloaded with pandas.
    mask_name = row[f"{target}_file"]
    if isinstance(mask_name, str) and mask_name:
        mask = tifffile.imread(root / mask_name) == 1
    else:
        mask = np.zeros(image.shape[1:], dtype=bool)
    return image, mask


class FirePatches(Dataset):
    def __init__(self, index, target=TARGET):
        self.index = index.reset_index(drop=True)
        self.target = target

    def __len__(self):
        return len(self.index)

    def __getitem__(self, position):
        image, mask = read_patch(self.index.iloc[position], self.target)
        image = torch.from_numpy(image.astype(np.float32) / MAX_PIXEL_VALUE)
        return image, torch.from_numpy(mask[None].astype(np.float32))


splits = {split: FirePatches(index[index.split == split]) for split in SPLIT_FRACTIONS}
for split, dataset in splits.items():
    print(f"{split}: {len(dataset)} patches")

In [ ]:
loader = DataLoader(splits["train"], batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(SEED))
images, masks = next(iter(loader))
print("images", tuple(images.shape), images.dtype, f"range {images.min():.3f} to {images.max():.3f}")
print("masks ", tuple(masks.shape), masks.dtype, "values", masks.unique().tolist())
print("fire pixel share in this batch", round(masks.mean().item(), 5))

In [ ]:
def false_color(image):
    """Stretch each channel to its 2nd to 98th percentile for display."""
    planes = []
    for plane in image.numpy():
        low, high = np.percentile(plane, (2, 98))
        planes.append(np.clip((plane - low) / max(high - low, 1e-6), 0, 1))
    return np.stack(planes, axis=-1)


count = min(4, len(images))
fig, axes = plt.subplots(2, count, figsize=(3 * count, 6), squeeze=False)
for column in range(count):
    axes[0, column].imshow(false_color(images[column]))
    axes[0, column].set_title(f"Bands {', '.join(map(str, INPUT_BANDS))}")
    axes[1, column].imshow(masks[column, 0], cmap="hot", vmin=0, vmax=1)
    axes[1, column].set_title(f"{TARGET}: {int(masks[column].sum())} fire px")
    for axis in axes[:, column]:
        axis.axis("off")
plt.tight_layout()

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Training device:", device)

## Patches in memory

Reading TIFFs from the Kaggle input mount every epoch would make it the bottleneck. Each split is read once, in parallel, and kept in memory as uint16 images and bool masks, about 7 GB for all of North America.

In [ ]:
PATCH_SIZE = 256


class CachedPatches(Dataset):
    def __init__(self, index, target=TARGET):
        rows = [row for _, row in index.iterrows()]
        self.images = np.empty((len(rows), len(CHANNELS), PATCH_SIZE, PATCH_SIZE), dtype=np.uint16)
        self.masks = np.empty((len(rows), 1, PATCH_SIZE, PATCH_SIZE), dtype=bool)
        with ThreadPoolExecutor(max_workers=16) as pool:
            for position, (image, mask) in enumerate(pool.map(lambda row: read_patch(row, target), rows)):
                self.images[position] = image
                self.masks[position, 0] = mask

    def __len__(self):
        return len(self.images)

    def __getitem__(self, position):
        image = torch.from_numpy(self.images[position].astype(np.float32) / MAX_PIXEL_VALUE)
        return image, torch.from_numpy(self.masks[position].astype(np.float32))


cached = {}
for split in SPLIT_FRACTIONS:
    started = time.time()
    cached[split] = CachedPatches(index[index.split == split])
    gigabytes = (cached[split].images.nbytes + cached[split].masks.nbytes) / 1e9
    print(f"{split}: {len(cached[split])} patches, {gigabytes:.2f} GB, {time.time() - started:.0f}s")

## Model

U-Net Light from [Pereira et al. 2021](https://arxiv.org/abs/2101.03409): four levels down, 16 filters in the first layer and twice as many at each level, batch normalization, and dropout at the bottom. It outputs one logit per pixel.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, in_channels, base=16, depth=4, dropout=0.1):
        super().__init__()
        widths = [base * 2**level for level in range(depth)]
        self.down = nn.ModuleList()
        channels = in_channels
        for width in widths:
            self.down.append(conv_block(channels, width))
            channels = width
        self.bottom = conv_block(widths[-1], widths[-1] * 2)
        self.dropout = nn.Dropout2d(dropout)
        self.up = nn.ModuleList(nn.ConvTranspose2d(width * 2, width, 2, stride=2) for width in reversed(widths))
        self.merge = nn.ModuleList(conv_block(width * 2, width) for width in reversed(widths))
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        skips = []
        for block in self.down:
            x = block(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.dropout(self.bottom(x))
        for up, merge, skip in zip(self.up, self.merge, reversed(skips)):
            x = merge(torch.cat([up(x), skip], dim=1))
        return self.head(x)


parameters = sum(parameter.numel() for parameter in UNet(len(INPUT_BANDS)).parameters())
print(f"U-Net Light: {parameters:,} parameters")

## Training

The paper's settings: binary cross-entropy, Adam at a learning rate of 0.001, batches of 16, up to 50 epochs, and a pixel is fire when the output is above 0.25.

After every epoch, val is scored with precision, recall, and IoU over all its fire pixels. The weights with the best val IoU are saved as `detection_unet.pt`, and training stops after `PATIENCE` epochs without improvement. Locally it runs 2 epochs, only to check the code.

Val IoU can collapse for a single epoch and recover the next, once to 0.016 at epoch 20. Only the best epoch is saved, so this does not reach the final model. Halving the learning rate when val IoU stalled did not stop it and scored lower on test, IoU 0.758 against 0.778, so the learning rate stays fixed.

In [ ]:
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
THRESHOLD = 0.25
MAX_EPOCHS = 50 if ON_KAGGLE else 2
PATIENCE = 5
WEIGHTS_PATH = OUTPUT_DIR / "detection_unet.pt"


@torch.no_grad()
def evaluate(model, dataset):
    """Loss, precision, recall, and IoU summed over every pixel in the split."""
    model.eval()
    losses, hits, predicted_pixels, actual_pixels, union = [], 0, 0, 0, 0
    for images, masks in DataLoader(dataset, batch_size=64):
        logits = model(images.to(device))
        masks = masks.to(device)
        losses.append(F.binary_cross_entropy_with_logits(logits, masks).item())
        predicted = torch.sigmoid(logits) > THRESHOLD
        actual = masks == 1
        hits += (predicted & actual).sum().item()
        predicted_pixels += predicted.sum().item()
        actual_pixels += actual.sum().item()
        union += (predicted | actual).sum().item()
    return {
        "loss": float(np.mean(losses)),
        "precision": hits / max(predicted_pixels, 1),
        "recall": hits / max(actual_pixels, 1),
        "iou": hits / max(union, 1),
    }


torch.manual_seed(SEED)
model = UNet(len(INPUT_BANDS)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_loader = DataLoader(cached["train"], batch_size=BATCH_SIZE, shuffle=True)

history, best, epochs_since_best = [], -1.0, 0
for epoch in range(1, MAX_EPOCHS + 1):
    started = time.time()
    model.train()
    losses = []
    for images, masks in train_loader:
        loss = F.binary_cross_entropy_with_logits(model(images.to(device)), masks.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    val = evaluate(model, cached["val"])
    history.append({"epoch": epoch, "train_loss": np.mean(losses), **{f"val_{k}": v for k, v in val.items()}})
    improved = val["iou"] > best
    print(
        f"epoch {epoch:2d}  train loss {np.mean(losses):.5f}  val loss {val['loss']:.5f}"
        f"  precision {val['precision']:.3f}  recall {val['recall']:.3f}  IoU {val['iou']:.3f}"
        f"  {time.time() - started:.0f}s{'  saved' if improved else ''}"
    )
    if improved:
        best, epochs_since_best = val["iou"], 0
        torch.save(model.state_dict(), WEIGHTS_PATH)
    else:
        epochs_since_best += 1
        if epochs_since_best >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs, stopping")
            break

pd.DataFrame(history).to_csv(OUTPUT_DIR / "detection_history.csv", index=False)

## Test scores

The best weights scored on the test split. These labels come from the fire algorithms, so the score shows how well the U-Net copies the voting mask. The hand-labeled set is the honest score and is not on Kaggle yet.

In [ ]:
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
results = pd.DataFrame({split: evaluate(model, cached[split]) for split in ("val", "test")}).T.round(4)
results.to_csv(OUTPUT_DIR / "detection_results.csv")
results

In [ ]:
busiest = cached["test"].masks.sum(axis=(1, 2, 3)).argsort()[::-1][:4].copy()
images = torch.stack([cached["test"][position][0] for position in busiest])
masks = torch.stack([cached["test"][position][1] for position in busiest])
with torch.no_grad():
    probabilities = torch.sigmoid(model(images.to(device))).cpu()

fig, axes = plt.subplots(3, len(busiest), figsize=(3 * len(busiest), 9), squeeze=False)
for column in range(len(busiest)):
    axes[0, column].imshow(false_color(images[column]))
    axes[0, column].set_title(f"Bands {', '.join(map(str, INPUT_BANDS))}")
    axes[1, column].imshow(probabilities[column, 0] > THRESHOLD, cmap="hot", vmin=0, vmax=1)
    axes[1, column].set_title(f"Predicted: {int((probabilities[column] > THRESHOLD).sum())} fire px")
    axes[2, column].imshow(masks[column, 0], cmap="hot", vmin=0, vmax=1)
    axes[2, column].set_title(f"{TARGET}: {int(masks[column].sum())} fire px")
for axis in axes.flat:
    axis.axis("off")
plt.tight_layout()